# 임베딩 모델 파인튜닝

- 임베딩 모델 파인튜닝은 사전 학습된 임베딩 모델을 특정 도메인이나 작업에 맞게 최적화하는 과정입니다.


## 1. 임베딩 모델의 학습 원리

- 의미가 비슷한 문장 쌍에는 높은 임베딩 유사도를, 의미가 다른 문장 쌍에는 낮은 유사도를 반환하도록 임베딩 벡터를 업데이트 하는 방식입니다.
- 임베딩 모델을 학습할 때는 의미가 유사한 문장 쌍과 유사하지 않은 문장 쌍을 대조하여 학습하는 방식, 즉 대조 학습을 활용합니다.


### 1.1 대조학습

- 포지티브 샘플
  - 의미적으로 관련이 있는 문장 쌍을 의미합니다.
  - 예: (기준 문서: "서울의 인구는?", 비교 문서: "서울의 인구는 약 970만명 입니다.")
- 네커티브 샘플
  - 기준 문서는 동일하지만, 비교 문서는 의미적으로 관련이 없거나 관련성이 낮은 문장을 준비하여 이들을 쌍으로 구성한 데이터입니다.
  - 예: (기준 문서: "서울의 인구는?", 비교 문서: "파리는 프랑스의 수도입니다.")


- 포지티브 샘플은 RAG를 수행할 때 사용자가 입력할 만한 검색어를 기준문서, 검색 결과로 유사도가 높게 나오기를 바라는 문서를 관련 있는 문서로 삼아 구성합니다.
- 네거티브 샘플은 RAG상황에서 같은 앵커에 대한 검색 결과에 포함되지 않기를 바라는 문서를 짝지어 구성합니다.


- 포지티브 샘플과 네거티브 샘플 구성
  - 기준 문서를 중심으로 유사도가 높은 쌍인 포지티브 샘플(관련 있는 쌍)과 유사도가 낮은 네거티브 샘플(관련 없는 쌍)을 모두 학습 데이터로 준비합니다.
- 대조 학습
  - 모델이 포지티브 샘플 쌍의 임베딩 간 거리는 가깝게, 네거티브 샘플 쌍의 임베딩 간 거리는 멀게 만들도록 학습합니다.
- 손실 함수 최적화
  - 임베딩 간 유사도를 계산하여, 포지티브 쌍의 임베딩 유사도는 높이고 네거티브 쌍의 임베딩 유사도는 낮추는 방향으로 손실 함수를 최적화합니다.


- 손실 함수는 모델이 예측한 결과와 실제 정답 간의 오차를 계산해 학습을 조정하는 기준이 됩니다.
- MultipleNegativesRankingLoss라는 손실 함수를 사용할 예정입니다.


### 1.2 데이터셋 구성

- 대조 학습에서는 하나의 기준 문서에 대해 하나의 포지티브 샘플과 하나 이상의 네거티브 샘플을 명시적으로 준비해야 합니다.
- 기준 문서는 앵커라고 부릅니다.
- 네거티브 샘플은 포지티브 샘플보다 양이 많을수록 좋습니다.


- 트리플렛 구성
  - 전통적인 방식은 각 학습 데이터를 (앵커, 포지티브, 네거티브) 형태의 트리플렛으로 구성하는 것입니다.


In [ ]:
# 전통적인 트리플렛 구성 예
triplets = [
    # (앵커, 포지티브, 네거티브)
    ("강아지를 기르는 방법", "반려견 양육 가이드", "고양이 사료 추천"),
    ("파이썬 코딩 튜토리얼", "파이썬 프로그래밍 기초", "자바스크립트 입문 강의"),
    # 수천, 수만 개의 트래플렛 필요
]

- 다중 네거티브 구성
  - 실제 모델 학습에서는 하나의 앵커에 여러개의 네거티브 샘플을 포함하는 구성이 더 효과적인 경우가 많습니다.


In [ ]:
# 다중 네거티브 샘플 구성 예
training_data = [
    {
        "anchor": "머신러닝이란?",
        "positive": "기계학습은 데이터로부터 패턴을 찾는 AI 기술입니다.",
        "negatives": [
            "오늘 날씨가 좋네요",
            "내일 회의는 2시에 시작합니다.이 식당의 불고기가 맛있습니다.",
            # 여러 개의 네거티브 샘플
        ],
    },
    # 수천개의 이러한 구조
]

- 다중 네거티브 구성은 학습 효과를 높일 수 있지만, 그만큼 데이터 준비의 난이도도 높아집니다.
- 임베딩 모델을 효과적으로 파인튜닝하기 위해서는, 특히 네거티브 샘플 선정이 가장 까다로운 작업중 하나입니다.


- 네거티브 샘플의 문서는 각 앵커와 관련 없는 텍스트여야만 합니다.
- 적절한 난이도의 네거티브 샘플을 선택해야 합니다.
  - 두 개의 쌍이 너무 관련이 없다면 임베딩 모델이 판단하기 너무 쉬워서 학습 효과가 거의 없게 됩니다.
  - 사람이 보아도 관련이 있는 것인지 관련이 없는 것인지 헷갈릴 정도의 문서 쌍이라면 난이도가 너무 높아져 학습에 오히려 방해가 됩니다.
- 다중 네거티브 샘플을 구성할 경우, 네거티브 샘플을 포지티브 샘플 대비 몇 배로 구성하느냐에 따라 데이터셋 크기가 기하급수적으로 증가하고, 만들어야 하는 데이터의 양이 많아지게 됩니다.


- 임베딩 파인튜닝에서 양질의 네거티브 샘플을 구성하는 것은 종종 전체 학습과정에서 가장 어려운 부분 중 하나입니다.


### 1.3 배치 내 네거티브 샘플링

- 배치 내에서 네거티브 샘플을 선정하는 학습 방법을 사용합니다.
- 이 방법은 명시적인 네거티브 샘플을 별도로 준비할 필요가 없다는 큰 장점이 있습니다.
- 학습 데이터에서 다른 앵커에서 사용하고 있는 샘플을 참고하여 자동으로 네거티브로 활용합니다.
- 이 원리를 이해하려면 배치라는 개념을 알아야 합니다.
- AI모델은 데이터를 적당한 개수의 묶음으로 나누어 학습합니다.
- 예를 들어 데이터가 5000개 이고 배치 크기를 40으로 설정했다면, 데이터를 40개씩 묶어 125회에 걸쳐 학습하게 됩니다.
- 배치란 모델이 한 번에 학습하는 데이터의 단위를 뜻하며, 병렬적으로 데이터를 몇개씩 학습할 것이냐를 의미합니다.


- 배치 내 네거티브 생성 방법은 포지티브 샘플만으로 데이터를 구성하더라도 배치내에서 네거티브 샘플들을 자동으로 만드는 학습 방법입니다.
- 사용자는 학습을 위해 포지티브 샘플만 제공하면 되며, 네거티브 샘플은 학습 시 배치내에서 자동으로 생성됩니다.


- 예를 들어 배치 크기가 4인 경우, 한번의 학습에 네 개의 서로 다른 앵커 문서와 그에 대응하는 포지티브 샘플이 사용되며, 이들 간 교차로 네거티브 샘플 역할도 동시에 수행됩니다.


```
배치 = [
    (앵커문서1, 문서1),
    (앵커문서2, 문서2),
    (앵커문서3, 문서3),
    (앵커문서4, 문서4)
]
```


- 앵커 문서1의 포지티브는 문서1, 나머지는 네거티브로 간주됩니다.
- 앵커 문서2의 포지티브는 문서2, 나머지는 네거티브로 간주됩니다.


- 하나의 배치 안에서 다른 쌍의 문서를 네거티브로 자동 활용하면서 대조 학습을 수행합니다.


```
[
    ("AI란 무엇인가?", "AI는 인간의 지능을 모방한 기술입니다."),
    ("딥러닝이란?", "신경망을 여러 층 쌓아 데이터로부터 학습하는 기계학습 방법입니다."),
    ("Python은 어디에 쓰이나요?", "Python은 데이터 분석, 웹 개발, AI 등에 널리 사용됩니다."),
    ("자연어 처리란?" , "컴퓨터가 인간의 언어를 이해하고 처리하는 AI의 한 분야입니다.")
]
```


### MultipleNegativesRankingLoss

- 이 손실 함수는 포지티브 샘플과의 유사도는 높이고, 네거티브 샘플과는 유사도는 낮추도록 설계되어 있습니다.


In [1]:
from sentence_transformers import SentenceTransformer, losses, InputExample
from torch.utils.data import DataLoader
import torch

# 모델 로드
model = SentenceTransformer("BAAI/bge-m3")

# 훈련 데이터 준비
train_examples = [
    InputExample(texts=["AI란 무엇인가?", "AI는 인간의 지능을 모방한 기술입니다."]),
    InputExample(
        texts=[
            "딥러닝이란?",
            "신경망을 여러 층 쌓아 데이터로부터 학습하는 기계학습 방법입니다.",
        ]
    ),
    InputExample(
        texts=[
            "Python은 어디에 쓰이나요?",
            "Python은 데이터 분석, 웹 개발, AI등에 널리 사용됩니다.",
        ]
    ),
    InputExample(
        texts=[
            "자연어 처리란?",
            "컴퓨터가 인간의 언어를 이해하고 처리하는 AI의 한 분야입니다.",
        ]
    ),
]

# 배치 크기가 클수록 성능이 향상될 수 있지만 GPU에 따라서 최대 비치 크기가 제한됨
batch_size = 32
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=batch_size)

# MultipleNegativesRankingLoss 설정
# 온도(temperature) 파라미터를 조정하여 손실 함수의 강도 조절 가능
loss = losses.MultipleNegativesRankingLoss(
    model, scale=20.0
)  # scale은 temperature의 역수

# 학습 설정
train_loss = losses.MultipleNegativesRankingLoss(model)
warmup_steps = int(len(train_dataloader) * 0.1)  # 전체 훈련 데이터의 10%

# 모델 학습
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=3,
    warmup_steps=warmup_steps,
    optimizer_params={"lr": 2e-5},
    output_path="./korean-sentence-embedding-model",
)

c:\workspace\python\rag_master\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\workspace\python\rag_master\.venv\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss


- 먼저 sentence_transformers 라이브러리를 통해 BAAI/bge-m3 모델을 기본 모델로 로드합니다.
- 이어 훈련데이터를 InputExample객체의 리스트로 준비합니다.
- 모델은 이러한 문장의 쌍들을 통해 유사한 문장들이 임베딩 공간에서 가깝게 위치하도록 학습합니다.
- DataLoader를 활용하여 배치크기 32로 데이터를 효율적으로 처리하도록 설정합니다.
  - 데이터가 4개밖에 없지만, 실제 상황에서는 데이터가 32개보다 많다고 가정합니다.
- 손실함수로 MultipleNegativesRankingLoss를 채택했습니다. 의미적으로 유사한 문장들은 가깝게, 그렇지 않은 문장들은 멀리 위치시키도록 모델을 유도합니다.
  - scale=20.0 파라미터는 온도의 역수로 손실함숨의 강도를 적절히 조절하는 역할을 합니다.
- 학습과정에서는 워밍업단계를 전체 훈련데이터의 10%로 설정합니다.
- 업데이트 하는 정도를 조절하는 학습률은 2e-5 로 지정합니다.
- model.fit() 함수를 호출하여 모델을 학습합니다.
- 학습횟수를 의미하는 에포크의 경우 3
- 완성된 모델은 korean-sentence-embedding-model 디렉터리에 저장합니다.


## 2 학습 시 성능을 높이는 방법

### 3.1 배치 크기 키우기

- 임베딩 모델을 효과적으로 학습시키려면 배치 크기를 크게 설정하는 것이 중요한 전략 중 하나입니다.
- 대조 학습은 동일 앵커 기준으로 네거티브 샘플이 포지티브 샘플보다 많을수록 학습 성능이 올라간다는 특징이 있습니다.
- 배치 크기가 4인 경우: 각 질문에 대해 3개의 네거티브 샘플
- 배치 크기가 32인 경우: 각 질문에 대해 31개의 네거티브 샘플
- 배치 크기는 GPU 메모리 용량에 따라 제한되므로 무한정 키울수는 없습니다.
- 구글 코랩에서 제공되는 무료 GPU를 사용할 경우 배치 크기는 3~4수준에 그치는 경우가 많습니다.


### 2.2 하드 네거티브 선정

- 더 어려운 네거티브 샘플, 하드 네거티브를 추가하면 성능을 더욱 향상시킬 수 있습니다.
- 하드네거티브는 명시적으로 사용자가 직접 선택하여 학습 데이터에 포함시키는 네거티브 샘플을 의미합니다.


In [ ]:
# 하드 네거티브 예제
train_examples = [
    # (앵커, 포지티브, 하드 네거티브 형태로 제공)
    InputExample(
        texts=[
            "AI란 무엇인가?",
            "AI는 인간의 지능을 모방한 기술입니다.",
            "AI는 로봇과 같은 물리적 형태를 가진 기계입니다.",
        ]
    ),
    InputExample(
        texts=[
            "딥러닝이란?",
            "신경망을 여러 층 쌓아 데이터로부터 학습하는 기계학습 방법입니다.",
            "컴퓨터가 스스로 생각하는 방법입니다.",
        ]
    ),
]

- 각 InputExample의 첫 번째 항목은 앵커(질문)입니다.
- 두 번째 항목은 포지티브 샘플(관련있는응답)입니다.
- 세 번째 이후 항목들은 하드 네거티브(관련 없지만 구분하기 어려운 응답)입니다.


- 손실함수는 각 쌍에 대해 다른 모든 앵커의 포지티브 샘플들과 모든 하드 네거티브 샘플들을 네거티브로 사용합니다.
- 포지티브와 유사도가 높아지도록 학습되고, 다른 앵커의 포지티브, 하드 네거티브와는 유사도가 낮아지도록 학습됩니다.


- 일반 네거티브(배치 내 무작위 네거티브)
  - 자동으로 배치 내에서 생성됨
  - 대부분 주제가 완전히 다른 무관한 문장들
  - 모델이 구분하기 상대적으로 쉬움
- 하드 네거티브(명시적 네거티브)
  - 사용자가 직접, 의도적으로 선택함
  - 포지티브와 주제는 유사하나 정확한 답변은 아님
  - 미묘한 의미 차이를 포함하여 모델에게 더 큰 도전이 됨
- 예시
  - 질문: 당뇨병의 증상은 무엇인가요?
  - 포지티브: 당뇨병의 주요 증상으로는 갈증 증가, 빈뇨, 체중 감소 등이 있습니다.
  - 하드 네거티브(명시적): 저혈당의 증상으로는 현기증, 발한, 불안감 등이 있습니다. (의료 관련 주제이지만 당뇨병이 아닌 저혈당에 관한 내용)
  - 일반 네거티브(배치 내 자동 선택): 파이썬은 객체지향 프로그래밍 언어입니다.(완전히 다른 주제)


- 하드 네거티브 샘플을 사용하면 모델이 더 미묘한 의미 차이를 학습하게 되어 정확도가 크게 향상될 수 있습니다.
- 가능하다면 일반 네거티브와 하드 네거티브를 병행해 사용하는 것이 이상적입니다.


### 2.3 그 외 학습 성능 향상을 위한 팁

- 학습 데이터와 실전과의 괴리 최소화
  - 학습에 사용할 데이터의 앵커는 실제 RAG에서 사용자가 입력할만한 질문으로 구성해야 합니다. 학습 데이터와 실제 RAG에서 입력될 질문의 차이가 클수록 학습 후의 효용은 떨어지기 마련입니다.
- 데이터 증강
  - 난이도가 높은 하드 네거티브 샘플은 충분히 확보하면 모델이 더 섬세한 의미 차이를 학습할 수 있어 성능 향상에 도움이 됩니다.
- 학습률 조정
  - 학습률을 바꿔가면서 여러 번 학습하여 모델의 성능을 평가하고, 최적의 학습률을 찾아보는 것이 좋습니다.
- 온도 파라미터 조정
  - 손실 함수의 scale파라미터를 조절하면 학습 강도를 세밀하게 조정할 수 있습니다.


## 3. 실전 파인튜닝

### 3.1 데이터 로드하기


In [1]:
import os
import requests
import json
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from openai import OpenAI
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, losses, InputExample
from sentence_transformers.evaluation import InformationRetrievalEvaluator
import torch
from sklearn.metrics.pairwise import cosine_similarity
import PyPDF2


- os: 환경 변수 설정에 사용되며, 임베딩 모델 파인튜닝 과정에서 오픈AI API키를 설정합니다.
- requests: PDF파일과 같은 학습 데이터를 인터넷에서 다운로드할 때 사용합니다.
- json: API 응답을 처리하거나 구성 설정을 저장/로드할 때 활용합니다.
- pandas: 임베딩 모델 성능 평가 결과를 데이터프레임으로 구성하고 분석하는 데 사용합니다.
- numpy: 벡터 연산을 수행하며 특히 임베딩 벡터 간 코사인 유사도 계산에 사용합니다.
- tqdm: 대용량 데이터셋을 처리할 때 진행 상황을 시각적으로 표시하여 학습 과정을 모니터링 합니다.
- OpenAI: GPT 모델을 사용해 문서로부터 질문을 생성하는 등의 작업에 활용합니다.
- DataLoader: 임베딩 모델 학습 시 배치 단위로 데이터를 효율적으로 로드합니다. 이때, 배치 크기는 성능에 큰 영향을 미칩니다.
- SentenceTransformer: 문장 임베딩 모델의 핵심 라이브러리로, 다양한 사전 학습 모델을 로드하고 파인튜닝합니다.
- losses: MultipleNegativesRankingLoss와 같은 손실함수를 제공하여 임베딩 모델이 관련 문서 쌍은 가깝게, 관련 없는 쌍은 멀게 학습하도록 합니다.
- InputExample: 파인튜닝용 학습데이터 포맷으로, 질문과 관련 문서 쌍을 모델이 이해할 수 있는 형태로 구성합니다.
- InformationRetrievalEvaluator: 파인튜닝된 모델의 검색 성능을 정확도, MRR, NDCG등 다양한 지표로 평가합니다.
- torch: 임베딩 모델의 기본 프레임워크로, 텐서 연산과 GPU 가속을 지원합니다.
- cosine_similarity: 임베딩 벡터 간 유사도를 계산하여 질문에 가장 관련성 높은 문서를 찾는 데 사용합니다.
- PyPDF2: PDF 파일을 읽는데 사용합니다.


In [2]:
from dotenv import load_dotenv

# .env 파일에서 환경 변수 로드
load_dotenv()
# 환경 변수에서 API 키 가져오기
api_key = os.getenv("OPENAI_API_KEY")

### 3.2 하드 네거티브 선정

- 깃허브 저장소에서 일본 ICT 동향 문서와 미국 ICT 동향 문서 두 가지를 다운로드합니다.
- 미국 ICT 동향 문서를 기준으로 임베딩 모델을 학습시키고, 동일한 도메인의 문서인 일본 ICT 동향 문서에 대해 검색 성능을 평가해보겠습니다.
- 실제 현업에서 임베딩 모델을 파인튜닝할 때도 실제 RAG에서 사용할 동일한 도메인의 데이터로 파인튜닝하면 더 좋은 효과를 얻을 수 있습니다.


In [4]:
# PDF 파일 다운로드
urls = [
    "https://raw.githubusercontent.com/langchain-kr/langchain-tutorial/main/Ch09.%20Embedding%20Fine-tuning/ict_japan_2024.pdf",
    "https://raw.githubusercontent.com/langchain-kr/langchain-tutorial/main/Ch09.%20Embedding%20Fine-tuning/ict_usa_2024.pdf",
]

for url in urls:
    filename = url.split("/")[-1]
    response = requests.get(url)
    with open(filename, "wb") as f:
        f.write(response.content)
    print(f"{filename} 다운로드 완료")

ict_japan_2024.pdf 다운로드 완료
ict_usa_2024.pdf 다운로드 완료


In [3]:
def extract_text_from_pdf(pdf_path):
    """PDF 파일에서 텍스트를 추출하는 함수"""
    text_chunks = []
    with open(pdf_path, "rb") as file:
        pdf_reader = PyPDF2.PdfReader(file)
        for page_num in range(len(pdf_reader.pages)):
            page = pdf_reader.pages[page_num]
            text = page.extract_text()
            # 페이지 단위로 청크 생성
            if text.strip():
                text = text.strip()
                # 문서 길이가 10자 초과인 경우만 추가
                if len(text) > 10:
                    text_chunks.append(text)
    return text_chunks


# 미국 ICT 동향(학습 데이터)
train_corpus = extract_text_from_pdf("ict_usa_2024.pdf")
print(f"학습 데이터 문서 개수: {len(train_corpus)}")

# 일본 ICT 동향(검증 데이터)
val_corpus = extract_text_from_pdf("ict_japan_2024.pdf")
print(f"검증 데이터 문서 개수: {len(val_corpus)}")

학습 데이터 문서 개수: 26
검증 데이터 문서 개수: 27


- 의미 있는 내용을 보장하기 위해 길이가 10자를 초과하는 텍스트만 청크에 추가합니다.


In [4]:
print("10번 문서:", train_corpus[10])

10번 문서: 13 Ⅰ. ICT 국가 산업 현황
 4.ICT 주요 법령 및 규제
  ② 반도체 과학법 (CHIPS and Science Act)
 반도체 ·전자 기업, $1,660 억 규모 투자 유치 
• 조 바이든 (Joe Biden) 미국 대통령은 2022년 7월 ‘반도체 과학법 (CHIPS and Science Act)’을 승인함  
• 반도체 과학법은 미국의 경쟁력을 강화하고 , 미국의 공급망을 탄력적으로 구축해 국가 안보를 
공고히 하며 국가의 주요 기술에 대한 접근을 지원하는 것을 목표로 함. 법률 제정으로 미국 내 
반도체 생산 제조사 관련 자본 투자는 25%의 세액 공제 혜택이 제공됨
• 미국 백악관은 2023년 8월, 반도체 과학법이 서명된 지 1년 만에 반도체와 전자 관련 기업들이 
1,660 억 달러(221조6,100 억 원)의 투자를 유치했다고 발표함 . 바이든 행정부의 집권 이후 기업들은 
미국 내 반도체와 전자 분야 투자에 총 2,310 억 달러(약 308조 3,850 억 원) 이상의 투자를 약속함
[표 8] 반도체 과학법 주요 이정표 및 진척 현황
주요 이정표 진척 현황 (23년 8월 기준)
미국 반도체 제조 
지원‣ 상무부는 CHIPS 통과 6개월 만에 해당 법에서 제공하는 390억 달러 반도체 제조 
인센티브에 대한 첫 번째 자금 조달 기회 시작
‣ 상무부는 42개 주에서 CHIPS 자금 지원에 관심 있는 460개 기업으로부터 소개서 접수
‣ 상무부는 CHIPS 인센티브 프로그램 관련 140명 이상의 인력으로 구성된 ‘칩스 포 
아메리카 (CHIPS for America)’ 팀 구성
‣ 재무부는 투자에 대한 25% 세액 공제 관련 지침 제공 위해 규칙 제안 발표
국가 안보를 
보호하고 동맹국 및 
파트너와 협력‣ 국무부는 국제 기술 보안 및 혁신 기금 시행 계획 발표
‣ 국방부와 상무부는 CHIPS 투자를 통한 안보 관련 반도체 제조를 위해 협력 확대 협의
‣ CHIPS 를 시행하면서 상무부는 여러 파트너 및 동맹국과 긴밀한 접촉

##


## 합성 데이터 생성

- 배치 인 네거티브 샘플 선정 방법을 사용하므로 네거티브 샘플을 따로 준비할 필요는 없습니다.
- gpt api를 이용하여 자동으로 포지티브 샘플을 만들어 보겠습니다.


In [5]:
# OpenAI 클라이언트 초기화
client = OpenAI()


# 각 문서에 대한 질문 생성(OpenAI API 사용)
def generate_queries(corpus, num_questions_per_chunk=2):
    all_queries = []
    all_positive_docs = []

    # 기본 프롬프트 템플릿 설정
    prompt_template = """
    다음은 참고할 내용입니다.

    ----------------------
    {context_str}
    ----------------------
    위 내용을 바탕으로 낼 수 있는 질문을 {num_questions_per_chunk}개 만들어 주세요.
    질문만 작성하고 실제 정답이나 보기 등은 작성하지 않습니다.

    해당 질문은 본문을 볼 수 없다고 가정합니다.
    따라서 '위 본문을 바탕으로~' 라는 식의 질문은 할 수 없습니다.

    질문은 아래와 같은 형식으로 번호를 나열하여 생성하십시오.

    1. (질문)
    2. (질문)
    """

    # corpus의 각 문서에 대해 반복 실행
    for text in tqdm(corpus):
        # 현재 문서에 대한 프롬프트 생성
        messages = [
            {
                "role": "system",
                "content": "You are a helpful assistant that generates questions based on provided content.",
            },
            {
                "role": "user",
                "content": prompt_template.format(
                    context_str=text, num_questions_per_chunk=num_questions_per_chunk
                ),
            },
        ]

        # GPT 모델을 사용해 질문 생성
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            temperature=0.7,
        )

        # 응답을 줄바꿈을 기준으로 분리하여 개별 질문으로 만듦
        result = response.choices[0].message.content.strip().split("\n")

        # 질문 형식 정리
        questions = []
        for line in result:
            if line.strip():
                parts = line.strip().split(". ", 1)
                if len(parts) > 1:
                    questions.append(parts[1])
                else:
                    questions.append(parts[0])

        # 빈 질문 제거
        questions = [q for q in questions if len(q) > 0]

        # 각 질문에 대해 문서 매칭 및 저장
        for question in questions:
            all_queries.append(question)
            all_positive_docs.append(text)

    return all_queries, all_positive_docs


- generate_queries 함수는 PDF에서 추출한 텍스트 데이터 train_corpus와 val_corpus를 입력 받아 처리합니다.
- 각 텍스트 문서마다 GPT모델에게 질문 생성을 요청합니다.
- 각 문서당 2개의 질문을 생성하도록 지정합니다.

### 예

- "2024년 일본 반도체 산업은 전년 대비 15% 성장했으며, 정부는 300억 엔의 추가 투자를 발표했다."
- "2024년 일본 반도체 산업의 성장률은 얼마인가?"
- "일본 정부가 반도체 산업에 발표한 추가 투자 금액은?"


- all_queries: 생성된 모든 질문이 들어 있습니다.
- all_positive_docs: 각 질문의 출처가 된 문서들이 순서대로 들어 있습니다.


- 합성 데이터를 생성하는 작업은 실제 실무에 적용할 때는 현재 구현하는 RAG 시스템의 실제 상황에서 사용자가 입력할 만한 질문들이 생성되도록 프롬프트를 조정하는 것이 바람직합니다.


In [6]:
# 학습 데이터 질문 생성
train_queries, train_positive_docs = generate_queries(train_corpus)
print(f"생성된 학습용 질문 개수: {len(train_queries)}")

# 검증 데이터 질문 생성
val_queries, val_positive_docs = generate_queries(val_corpus)
print(f"생성된 검증용 질문 개수: {len(val_queries)}")

  0%|          | 0/26 [00:00<?, ?it/s]

생성된 학습용 질문 개수: 52


  0%|          | 0/27 [00:00<?, ?it/s]

생성된 검증용 질문 개수: 54


- train_queries에는 '미국 ICT 동향 문서'로부터 만들어낸 26개의 문서에 해당하는 train_corpus를 입력하여 만든 질문들이 저장되어 있습니다.
- 2개씩 생성하도록 지정했기 때문에 총 52개의 질문이 만들어졌습니다.
- 같은 원리로 val_queries는 '일본 ICT 동향 문서' 27개이므로 총 54개의 포지티브 샘플입니다.
- 이제 이 데이터를 학습에 사용할 수 있도록 InputExample 객체의 리스트 형태로 변환합니다.


In [7]:
# 학습 데이터 준비
train_examples = []
for query, doc in zip(train_queries, train_positive_docs):
    example = InputExample(texts=[query, doc])
    train_examples.append(example)

In [8]:
# 첫번째 데이터 출력
train_examples[0].texts

['ICT 관련 주요 정책과 법령에는 어떤 내용이 포함되어 있는가?',
 'Ⅰ ICT국가산업현황  4\n(*) SUMMARY\n1. 국가 개황\n2. ICT 정부기구\n3. ICT 주요정책\n4. ICT 주요법령및규제\n5. ICT 주요기업\n6. 한국 협력 및 국내기업 진출사례\nⅡ ICT이슈Top 10  16\n(*) SUMMARY\n① 미국 빅테크 기업, 인공지능 챗봇 개발에 주력\n② 미국, 일본과 양자컴퓨팅 개발 협력\n③ 미국, 우주 클라우드 컴퓨팅 시장 주도\n④ 미국, 드론 배송 도입 활발\n⑤ 미국, 긍정적인 의료 AI 인식 바탕으로 연구 활발\n⑥ 미국, 반도체 산업 활성화에 박차\n⑦ 미국, 기술 교류를 위한 국가 간 협력 활발\n⑧ 미국, 사이버 보안 대응 강화\n⑨ 미국, 6G 주도권 확보 위한 연구 추진\n⑩ 미 국방부 , 디지털 트윈 기술 도입 확대\n※ 참고문헌']

- 출력 결과를 보면 두 개의 원소를 지닌 리스트가 출력되는데, 각각 생성한 질문과 질문을 생성하기 위해 참고한 문서입니다.
- 이 둘은 서로 연관이 있는 질문과 문서이므로 포지티브 샘플 관계입니다.
- 두 번째 데이터를 출력해 봅시다.


In [9]:
# 첫번째 데이터 출력
train_examples[1].texts

['미국의 빅테크 기업들이 최근 집중하고 있는 기술 개발 분야는 무엇인가?',
 'Ⅰ ICT국가산업현황  4\n(*) SUMMARY\n1. 국가 개황\n2. ICT 정부기구\n3. ICT 주요정책\n4. ICT 주요법령및규제\n5. ICT 주요기업\n6. 한국 협력 및 국내기업 진출사례\nⅡ ICT이슈Top 10  16\n(*) SUMMARY\n① 미국 빅테크 기업, 인공지능 챗봇 개발에 주력\n② 미국, 일본과 양자컴퓨팅 개발 협력\n③ 미국, 우주 클라우드 컴퓨팅 시장 주도\n④ 미국, 드론 배송 도입 활발\n⑤ 미국, 긍정적인 의료 AI 인식 바탕으로 연구 활발\n⑥ 미국, 반도체 산업 활성화에 박차\n⑦ 미국, 기술 교류를 위한 국가 간 협력 활발\n⑧ 미국, 사이버 보안 대응 강화\n⑨ 미국, 6G 주도권 확보 위한 연구 추진\n⑩ 미 국방부 , 디지털 트윈 기술 도입 확대\n※ 참고문헌']

- 동일한 문서에 대해 질문이 2개 이므로 질문은 동일합니다.


## 3.4 모델 로드하기

- 배치 크기, 학습할 모델, 사용할 손실 함수를 설정합니다.
- 배치 크기를 4로 설정하겠습니다.
- DataLoader() 에 train_examples를 전달하고 batch_size값을 4로 설정합니다.
- 총 54개의 학습데이터는 4개씩 묶여 배치 단위로 처리됩니다.
- 일반적으로 배치 크기가 클수록 더 많은 네거티브 샘플이 생성되므로 성능이 더 좋아질 수 있습니다.


In [10]:
BATCH_SIZE = 4  # 배치 크기 조정
loader = DataLoader(train_examples, batch_size=BATCH_SIZE, shuffle=True)

- 이제 학습에 사용할 모델을 선택합니다.
- SentenceTransformer 모듈을 사용하여 허깅페이스 저장소로부터 모델을 다운로드 합니다.


In [11]:
# 모델 설정
model_id = "BAAI/bge-m3"
model = SentenceTransformer(model_id)

- 손실 함수로는 MultipleNegativesRankingLoss를 사용합니다.
- 포지티브 샘플과의 유사도는 높일수록 좋습니다. 즉, 검색어(앵커)와 관련 있는 문서가 가까이 있도록 학습합니다
- 네거티브 샘플과의 유사도는 낮을수록 좋습니다. 즉, 관련 없는 문서는 멀어지도록 학습합니다.


In [12]:
# 손실 함수 설정
loss = losses.MultipleNegativesRankingLoss(model)

## 3.5 평가 데이터 전처리

- 검색 성능을 평가하는데 InformationRetrievalEvaluator를 사용합니다.
- 이 도구를 사용하려면 평가 데이터를 특정 형식으로 전처리해야 합니다.
- InformationRetrievalEvaluator는 정보 검색 모델을 평가하는 도구로 3가지 필수 데이터 구조를 입력 받습니다.


### 1. queries

- 질문 ID를 키로, 질문 텍스트를 값으로 갖는 파이썬 딕셔너리입니다.

```json
{
  "q1": "인공지능의 정의는 무엇인가?",
  "q2": "머신러닝과 딥러닝의 차이점은?",
  "q3": "자연어 처리란 무엇인가?"
}
```


### 2. corpus

- 문서 ID를 키로, 문서 텍스트를 값으로 갖는 파이썬 딕셔너리입니다.

```json
{
  "d1": "인공지능(AI)은 인간의 학습, 추론, 결정 능력 등을 컴퓨터 시스템으로 구현한 기술이다. 인공지능은 머신러닝, 딥러닝 등 다양한 하위 분야를 포함한다.",
  "d2": "머신러닝은 컴퓨터가 데이터로부터 패턴을 학습하여 예측이나 의사결정을 수행하는 기술이다. 반면 딥러닝은 인공 신경망을 활용하여 더 복잡한 패턴을 학습하는 머신러닝의 한 분야이다.",
  "d3": "자연어 처리(NLP)는 컴퓨터가 인간의 언어를 이해, 해석, 생성할 수 있도록 하는 인공지능의 한 분야이다. 기계번역, 감성분석, 텍스트 요약 등의 응용이 있다."
}
```


### 3. relevant_docs

- 질문ID를 키로, 해당 질문에 관련된 문서 ID들의 집합(set)을 값으로 갖는 파이썬 딕셔너리입니다.

```python
{
    "q1": set(["d1"]),  #인공지능 질문은 d1 문서와 관련
    "q2": set(["d2"]),  #머신러닝/딥러닝 질문은 d2 문서와 관련
    "q3": set(["d3"])   #자연어 처리 질문은 d3 문서와 관련
}
```


- 실제 코드를 가정하여 구체적으로 예를 들어 살펴보겠습니다.

```python
# 기존에 전처리한 형태
val_queries = [
    "인공지능의 정의는 무엇인가?",
    "머신러닝과 딥러닝의 차이점은?",
    "자연어 처리란 무엇인가?",
]
val_positive_docs = [
    "인공지능(AI)은 인간의 학습, 추론, 결정 능력 등을 컴퓨터 시스템으로 구현한 기술이다. 인공지능은 머신러닝, 딥러닝 등 다양한 하위 분야를 포함한다.",
    "머신러닝은 컴퓨터가 데이터로부터 패턴을 학습하여 예측이나 의사결정을 수행하는 기술이다. 반면 딥러닝은 인공 신경망을 활용하여 더 복잡한 패턴을 학습하는 머신러닝의 한 분야이다.",
    "자연어 처리(NLP)는 컴퓨터가 인간의 언어를 이해, 해석, 생성할 수 있도록 하는 인공지능의 한 분야이다. 기계번역, 감성분석, 텍스트 요약 등의 응용이 있다.",
]

# InformationRetrievalEvaluator를 위한 변환
val_dataset = {
    "queries": {
        "q1": "인공지능의 정의는 무엇인가?",
        "q2": "머신러닝과 딥러닝의 차이점은?",
        "q3": "자연어 처리란 무엇인가?",
    },
    "corpus": {
        "d1": "인공지능(AI)은 인간의 학습, 추론, 결정 능력 등을 컴퓨터 시스템으로 구현한 기술이다. 인공지능은 머신러닝, 딥러닝 등 다양한 하위 분야를 포함한다.",
        "d2": "머신러닝은 컴퓨터가 데이터로부터 패턴을 학습하여 예측이나 의사결정을 수행하는 기술이다. 반면 딥러닝은 인공 신경망을 활용하여 더 복잡한 패턴을 학습하는 머신러닝의 한 분야이다.",
        "d3": "자연어 처리(NLP)는 컴퓨터가 인간의 언어를 이해, 해석, 생성할 수 있도록 하는 인공지능의 한 분야이다. 기계번역, 감성분석, 텍스트 요약 등의 응용이 있다.",
    },
    "relevant_docs": {
        "q1": set(["d1"]),  # 인공지능 질문은 d1 문서와 관련
        "q2": set(["d2"]),  # 머신러닝/딥러닝 질문은 d2 문서와 관련
        "q3": set(["d3"]),  # 자연어 처리 질문은 d3 문서와 관련
    },
}
```


In [13]:
# 평가 데이터셋 구성
val_dataset = {"queries": {}, "corpus": {}, "relevant_docs": {}}

# 문서 ID를 먼저 생성
doc_ids = {}
for i, doc in enumerate(val_corpus):
    doc_id = f"q{i}"
    val_dataset["corpus"][doc_id] = doc
    doc_ids[doc] = doc_id

# 질문에 ID를 부여하고 관련 문서 설정
for i, (query, doc) in enumerate(zip(val_queries, val_positive_docs)):
    query_id = f"q{i}"
    val_dataset["queries"][query_id] = query

    # 해당 질문이 어떤 문서에서 왔는지 찾기
    doc_id = doc_ids[doc]

    # 관련 문서 설정
    if query_id not in val_dataset["relevant_docs"]:
        val_dataset["relevant_docs"][query_id] = set()
    val_dataset["relevant_docs"][query_id].add(doc_id)

# 검증 데이터 셋 설정: 평가를 위한 쿼리, 문서, 정답 문서 목록
dataset = val_dataset

- 이제 dataset에는 InformationRetrievalEvaluator를 사용하는데 필요한 데이터인 queries, corpus, relevant_doc가 저장되어 있습니다.


In [14]:
dataset["corpus"]

{'q0': 'Ⅰ ICT국가산업현황  4\n(*) SUMMARY\n1. 국가 개황\n2. ICT 정부기구\n3. ICT 주요정책\n4. ICT 주요법령및규제\n5. ICT 주요기업\n6. 한국 협력 및 국내기업 진출사례\nⅡ ICT이슈Top 10  16\n(*) SUMMARY\n① 일본, 아시아에서 두 번째로 큰 데이터센터 허브\n② 일본, 자체 개발 소프트웨어로 사이버보안 강화\n③ 일본, Web3 산업 성장 촉진 도모\n④ 일본, 정부 행정 업무에 생성형 AI 도입\n⑤ 일본, 첫 자체 제작 양자컴퓨터 공개\n⑥ 일본, 6G 기술 강화 위해 협력 및 규제 완화\n⑦ 일본, 레벨 4 자율주행 허용\n⑧ 일본, 생체인식 결제 도입 증가\n⑨ 일본, 행정 서비스 디지털화 노력\n⑩ 일본, 인재 부족으로 디지털 인력 강화에 힘써\n※ 참고문헌',
 'q1': 'Ⅰ ICT 국가 산업 현황                   4\n   (*) SUMMARY\n   1. 국가 개황\n   2. ICT 정부기구\n   3. ICT 주요 정책\n   4. ICT 주요 법령 및 규제\n   5. ICT 주요기업\n   6. 한국 협력 및 국내기업 진출사례',
 'q2': '5 Ⅰ. ICT 국가 산업 현황\n 1.국가 개황\n 일본, 글로벌 혁신지수 세계 40위\n• 일본의 인터넷 사용자 비중은 82.9% 이며 고정 광대역 가입자 비중은 36.0% 임\n• 일본 글로벌 혁신지수는 세계 13위임. ‘인프라 ’ 및 ‘지식 및 기술 생산’ 지표가 상대적으로 \n우위에 있으며 , ‘창조적 생산’이 상대적 열위를 보임\n 기시다 총리, 결제 활성화 정책에 초점\n• 2023년 일본 물가상승률은 3.1%로 41년에 최대폭을 기록함 . 일본은행은 2023년 12월 \n금융정책결정회의에서 단기금리 목표를 △0.1%로 동결하면서 마이너스 금리 종료를 보류함\n• 일본 기시다 후미오 (岸⽥⽂雄 ) 총리는 향후 경제 활성화를 위한 정책에 초점을 맞출 것임을 밝힘. \n그중에서도 감세 논의에 속도를 

In [15]:
dataset["queries"]

{'q0': '일본의 데이터센터 허브 규모는 아시아에서 어느 정도인가요?',
 'q1': '일본에서 정부 행정 업무에 도입된 새로운 기술은 무엇인가요?',
 'q2': 'ICT 정부기구는 어떤 역할을 수행하며, 그 중요성은 무엇인가요?',
 'q3': '한국의 ICT 주요기업에는 어떤 기업들이 있으며, 그들의 특징은 무엇인가요?',
 'q4': '일본의 인터넷 사용자 비중과 고정 광대역 가입자 비중은 각각 얼마인가요?',
 'q5': '기시다 총리는 어떤 경제 정책에 중점을 두고 있으며, 그 중 하나로 어떤 논의에 속도를 내고 있나요?',
 'q6': '일본 총무성(MIC)의 주요 역할 중 하나는 무엇인가요?',
 'q7': '최근 일본 총무성이 발간한 가이드라인의 목적은 무엇인가요?',
 'q8': '일본 문부과학성(MEXT)의 주요 역할 중 하나로, 과학기술 개발을 담당하는 기구는 무엇인가요?',
 'q9': '최근 문부과학성이 추진하고 있는 교육 부문 DX(Data experience) 관련 사업의 이름은 무엇인가요?',
 'q10': '일본 경제산업성이 최근 발표한 국제 표준의 목적은 무엇인가?',
 'q11': '일본에서 레벨 4 자율주행 모빌리티 서비스가 시작된 프로젝트는 어떤 부처와 협업하여 진행되었는가?',
 'q12': '일본 디지털청의 주요 목표는 무엇인가요?',
 'q13': '일본 디지털청에서 추진하고 있는 정책 중 하나는 무엇인가요?',
 'q14': '생성형 AI 서비스 사용 시 개인정보 취급 사업자가 유의해야 할 사항은 무엇인가요?',
 'q15': '일본 개인정보보호위원회가 발표한 생성형 AI 서비스 사용 지침의 주요 내용은 어떤 것들이 있나요?',
 'q16': '일본 내각부가 AI 모델 성능 향상을 위해 추진하는 주요 정책은 무엇인가요?',
 'q17': 'AI 학습 데이터 제공 촉진을 위한 액션플랜에서 기계 판독이 불가능한 데이터의 변환을 위해 어떤 노력이 이루어지고 있나요?',
 'q18': '일본의 반도체 및 디지털 산업 

In [16]:
dataset["relevant_docs"]

{'q0': {'q0'},
 'q1': {'q0'},
 'q2': {'q1'},
 'q3': {'q1'},
 'q4': {'q2'},
 'q5': {'q2'},
 'q6': {'q3'},
 'q7': {'q3'},
 'q8': {'q4'},
 'q9': {'q4'},
 'q10': {'q5'},
 'q11': {'q5'},
 'q12': {'q6'},
 'q13': {'q6'},
 'q14': {'q7'},
 'q15': {'q7'},
 'q16': {'q8'},
 'q17': {'q8'},
 'q18': {'q9'},
 'q19': {'q9'},
 'q20': {'q10'},
 'q21': {'q10'},
 'q22': {'q11'},
 'q23': {'q11'},
 'q24': {'q12'},
 'q25': {'q12'},
 'q26': {'q13'},
 'q27': {'q13'},
 'q28': {'q14'},
 'q29': {'q14'},
 'q30': {'q15'},
 'q31': {'q15'},
 'q32': {'q16'},
 'q33': {'q16'},
 'q34': {'q17'},
 'q35': {'q17'},
 'q36': {'q18'},
 'q37': {'q18'},
 'q38': {'q19'},
 'q39': {'q19'},
 'q40': {'q20'},
 'q41': {'q20'},
 'q42': {'q21'},
 'q43': {'q21'},
 'q44': {'q22'},
 'q45': {'q22'},
 'q46': {'q23'},
 'q47': {'q23'},
 'q48': {'q24'},
 'q49': {'q24'},
 'q50': {'q25'},
 'q51': {'q25'},
 'q52': {'q26'},
 'q53': {'q26'}}

- 이제 dataset 을 InformationRetrievalEvaluator에 전달합니다.


In [17]:
# 검증 데이터셋에서 코퍼스(전체문서), 쿼리, 기리고 각 쿼리와 관련된 문서 가져오기
corpus = dataset["corpus"]
queries = dataset["queries"]
relevant_docs = dataset["relevant_docs"]

evaluator = InformationRetrievalEvaluator(
    queries=queries, corpus=corpus, relevant_docs=relevant_docs
)

- 테스트 데이터를 이용하기 위해 평가하기 위한 준비가 끝났습니다.
- 파인튜닝을 진행하고 학습 전, 후 모델에 대해 평가를 진행해보겠습니다.


## 3.6 모델 학습하기

- 실제로 모델을 학습시켜 봅시다. 데이터 양이 적기 때문에 많은 학습이 필요하지는 않습니다.
- 학습 횟수를 의미하는 EPOCHS값을 2로 설정합니다.
- 학습 데이터 52개에 대해 총 2회 반복하여 학습합니다.


In [18]:
EPOCHS = 2

# W&B(WandB, Weights and Biases) 로깅 비활성화
# W&B는 학습 과정을 실시간으로 추적하고 시각화할 수 있는 도구
os.environ["WANDB_DISABLED"] = "true"

# 학습 초기에 학습률을 점진적으로 증가시키는 단계 수 설정
# 전체 학습 단계의 10%를 워밍업으로 사용
warmup_steps = int(len(loader) * EPOCHS * 0.1)

# 모델 학습
model.fit(
    train_objectives=[(loader, loss)],  # 학습 데이터 로더와 손실 함수 설정
    epochs=EPOCHS,  # 총 에포크 수
    warmup_steps=warmup_steps,  # 워밍업단계
    output_path="exp_finetune",  # 학습된 모델 저장 경로
    show_progress_bar=True,  # 학습 진행률 표시 여부
)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

c:\workspace\python\rag_master\.venv\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss


- 코드에서 os.environ["WANDB_DISABLED"] = "true" 는 학습 시 로그를 기록하는 모듈을 여기서는 사용하지 않는다는 의미입니다.
- 학습 과정을 시각화하지 않고 간단히 진행할 때 유용합니다.
- warmup_steps는 학습 초기에 학습률을 점진적으로 증가시키는 단계 수 입니다.
- 데이터 로더의 길이와 에포크 수를 곱한 값의 10%를 사용하므로, 전체 학습의 초반 10%는 학습률이 서서히 증가하게 됩니다.
- 실제 학습은 model.fit() 함수를 통해 이루어집니다.
- 준비한 데이터 로더와 손실 함수를 연결합니다.
- output='exp_finetune'은 파인튜닝된 모델을 exp_finetune 디렉터리에 저장한다는 의미입니다.


# 3.7 검색 성능 평가 지표

- InformationRetrievalEvaluator 에서 사용하는 총 6개의 주요 평가 지표를 간단히 정리합니다.


### 1. Accuracy

- Accuracy는 정답이 상위 몇 개의 검색 결과 안에 포함되었는지를 평가하는 지표입니다.
- 중요한 점은 정답이 포함되기만 하면 성공으로 간주한다는 것입니다.
- 예를들어 Accuracy@5가 0.92라는 값은 전체 질문중 약 92%에서 상위 5개의 결과 안에 정답이 하나라도 포함되었다는 뜻입니다.
- Accuracy는 검색 시스템이 얼마나 자주 정답을 포함하는지를 측정하며, 정답의 개수나 위치는 고려하지 않습니다.
- 질문 1: [정답, 오답, 오답, 오답, 오답] -> 포함(성공)
- 질문 2: [오답, 오답, 정답, 오답, 오답] -> 포함(성공)
- 질문 3: [오답, 오답, 오답, 오답, 오답] -> 미포함(실패)
- 이때 Accuracy@5 = 2/3 -> 0.667(66.7%) 입니다.


### 2. Precision

- Precision은 상위 검색 결과가 '얼마나 정확히 정답으로 이루어져 있는가?'를 평가합니다.
- Precision@k는 상위 k개의 검색 결과 중 정답이 차지하는 비율입니다.
- 예를들어, Precision@5가 0.20이라는 값은 상위 5개의 결과 중 평균적으로 20%가 정답이라는 뜻입니다.
- Precision은 검색 결과가 불필요한 정보를 얼마나 적게 포함하고 있는지를 보여줍니다.
- 질문 1: [정답, 오답, 오답, 오답, 오답] => Precision@5 = 1/5 = 0.2
- 질문 2: [오답, 정답, 오답, 오답, 오답] => Precision@5 = 1/5 = 0.2
- 질문 3: [오답, 오답, 오답, 오답, 오답] => Precision@5 = 0/5 = 0.0
- 평균 Precision@5 = (0.2 + 0.2 + 0.0) / 3 =0.133(13.3%) 입니다.


### 3. Recall

- Recall은 검색 결과가 얼마나 '포괄적으로' 정답을 포함하고 있는지를 평가합니다.
- Recall@k는 전체 정답 중 검색 결과 상위 k개 안에 포함된 정답의 비율을 나타냅니다.
- Recall은 정답을 놓치지 않고 얼마나 잘 찾아내는지를 보여줍니다.
- 참고로 실습데이터에서는 각 질문당 정답이 하나씩만 있기 때문에 Recall과 Accuracy의 값이 동일합니다.
- 예를 들어, 질문 하나에 정답이 2개 있다고 가정합시다.
- 질문 1: [정답, 정답, 오답, 오답, 오답] => Recall@5 = 2/2 = 1.0
- 질문 2: [정답, 오답, 오답, 오답, 오답] => Recall@5 = 1/2 = 0.5
- 질문 3: [오답, 오답, 오답, 오답, 오답] => Recall@5 = 0/2 = 0.0
- 평균 Recall@5 = (1.0 + 0.5 + 0.0) / 3 = 0.5 (50%) 입니다.


### 4 NDCG(Normalized Discounted Cumulative Gain)

- NDCG는 검색 결과에서 정답이 높은 순위에 배치될수록 높은 점수를 부여합니다.
- 이는 단순히 정답이 포함되었는지를 넘어, 정답의 순위가 사용자에게 얼마나 유용한지를 평가하는 지표입니다.
- NDCG@10이 0.85라는 값은 정답이 대체로 높은 순위에 배치되었음을 의미합니다.
- 질문 1: [정답, 정답, 오답] -> NDCG = 1.0(정답이 모두 상위에 있음)
- 질문 2: [오답, 정답, 오답] -> NDCG는 약 0.63(정답이 두번째 위치에 있음)
- 질문 3: [오답, 오답, 정답] -> NDCG는 약 0.39(정답이 세번째 위치에 있음)
- 평균 NDCG@3 = (1.0+0.63+0.39)/3 = 0.673 입니다.


### 5. MRR(Mean Reciprocal Rank)

- MRR은 정답이 처음 등장한 순위의 역수를 평균한 값입니다.
- 각 질문에 대해 정답이 검색 결과에서 처음 등장한 순위의 역수를 계산한 후, 모든 질문에 대해 그 값을 평균냅니다.
- MRR@10에서 @10은 상위 10개의 검색 결과까지만 고려한다는 의미입니다.
- 정답이 11위 이후에 등장하면 해당 질문은 계산에서 제외되거나 Reciprocal Rank는 0으로 간주됩니다.
- 이 지표는 사용자가 정답을 얼마나 빠르게 찾을 수 있는지를 평가합니다.
- MRR값이 높을수록, 정답이 더 상위 순위에 배치되어 있다는 의미입니다.
- 질문 1:[정답, 오답, 오답, 오답, ...] Reciprocal Rank = 1/1 = 1.0 (정답이 1위에 있음)
- 질문 2:[오답, 정답, 오답, 오답, ...] Reciprocal Rank = 1/2 = 0.5 (정답이 2위에 있음)
- 질문 3:[오답, 오답, 정답, 오답, ...] Reciprocal Rank = 1/3 = 0.33 (정답이 3위에 있음)
- 질문 4:[오답, 오답, 오답, ...] Reciprocal Rank = 0 (정답이 상위 10위 안에 없음)
- 평균 MRR@10 = (1.0 + 5.0 + 0.33 + 0)/4 = 0.458 입니다.


### 6. MAP(Mean Average Precision)

- MAP는 각 정답을 찾을 때마다의 Precision 값을 계산하여 평균을 낸 값으로, 검색 결과의 전반적인 정확도와 일관성을 평가합니다.
- MAP@100에서 @100은 검색 겨로가의 상위 100개 항목까지만 Precision 값을 계산한다는 의미입니다.
- 정답이 101위 이후에 있다면 해당 정답은 계산에서 제외됩니다.
- 이는 평가 범위를 제한함으로써 특정 상위 결과 내에서의 성능을 측정합니다.
- [정답, 정답, 오답, 오답, 정답]
- 첫 번째 정답을 찾았을때: Precision@1 = 1/1 = 1.0
- 두 번째 정답을 찾았을때: Precision@2 = 2/2 = 1.0
- 세 번째 정답을 찾았을때: Precision@5 = 3/5 = 0.6
- MAP@5 = (1.0 + 1.0 + 0.6)/3 = 0.867입니다.
- MAP@100 = 0.818이라는 값은 상위 100개의 검색 결과 내에서 정답을 찾을 때마다 계산된 Precision 값의 평균이 0.818이라는 뜻입니다.
- 이는 시스템이 상위 100개의 결과에서 정답을 얼마나 정확하고 일관되게 제공하는지를 평가하는 지표입니다.


## 3.8 파인튜닝 모델 평가하기

1. 모델은 queries 딕셔너리의 각 질문과 corpus 딕셔너리의 모든 문서를 임베딩 벡터로 변환합니다.
2. 각 질문 벡터와 모든 문서 벡터간의 코사인 유사도 점수를 계산합니다.
3. 각 질문마다 문서들을 유사도 점수가 높은 순서로 정렬합니다.
4. relevant_docs 딕셔너리에 명시된 정답 문서들이 이 정렬된 리스트에서 어떤 순위에 있는지 확인합니다.
5. MRR(Mean Reciprocal Rank), NDCG(Normalized Discounted Cumulative Gain), Percision@k Recall@k등의 검색 성능 평가 지표를 계산합니다.


- 예를 들어 "인공지능이란?" 질문("q0")에 대해 모델이 문서들을 다음과 같이 순위를 매겼다고 가정해봅시다

1. "d0"(인공지능 문서): 0.95점
2. "d2"(자연어 처리 문서): 0.70점
3. "d1"(머신러닝 문서): 0.60점


- relevant_docs 딕셔너리에 "q0": {"d0"}이 있다면, 이는 질문 "q0"의 정답 문서가 "d0"임을 의미합니다.
- 모델이 "d0"을 1위로 정확히 찾았으므로 이 질문에 대해서는 좋은 성능을 보인것입니다.
- 현재의 임베딩 모델은 앞서 설명한 검색 성능 평가 지표에 따라 높은 점수를 얻게 됩니다.


In [19]:
def evaluate_st(dataset, model_id, name, evaluator):
    """
    SentenceTransformer 모델의 검색 성능을 평가하는 함수
    """
    # 평가 결과를 저장할 디렉터리 생성
    os.makedirs("results", exist_ok=True)

    # 평가할 SentenceTransformer 모델 로드
    model = SentenceTransformer(model_id)

    # 모델 평가 수행
    result = evaluator(model)

    # 결과를 DataFrame으로 변환하고 CSV로 저장
    result_df = pd.DataFrame([result]) if isinstance(result, dict) else result
    output_path = f"results/Information-Retrieval_evaluation_{name}_results.csv"
    result_df.to_csv(output_path, index=False)

    return result


- evaluate_st() 함수는 학습하고자 하는 모델과 앞서 구현한 evaluator를 전달하면 해당 모델에 대한 평가 결과를 기록하고 results 디렉터리 안에 CSV파일로 저장합니다.
- 이 CSV 파일에는 상세한 평가 결과가 기록됩니다.
- 다음은 원본 모델과 파인튜닝 모델에 대해 각각 evaluate_st() 함수를 호출하여 평가를 진행하는 코드입니다.


In [20]:
# 원본 모델 평가
original_model_path = "BAAI/bge-m3"
evaluate_st(
    dataset=val_dataset,
    model_id=original_model_path,
    name="original",
    evaluator=evaluator,
)

# 파인튜닝된 모델 평가
finetuned_model_path = "exp_finetune"
evaluate_st(
    dataset=val_dataset,
    model_id=finetuned_model_path,
    name="finetuned",
    evaluator=evaluator,
)

{'cosine_accuracy@1': 0.7962962962962963,
 'cosine_accuracy@3': 0.9444444444444444,
 'cosine_accuracy@5': 0.9629629629629629,
 'cosine_accuracy@10': 1.0,
 'cosine_precision@1': 0.7962962962962963,
 'cosine_precision@3': 0.31481481481481477,
 'cosine_precision@5': 0.1925925925925925,
 'cosine_precision@10': 0.09999999999999996,
 'cosine_recall@1': 0.7962962962962963,
 'cosine_recall@3': 0.9444444444444444,
 'cosine_recall@5': 0.9629629629629629,
 'cosine_recall@10': 1.0,
 'cosine_ndcg@10': 0.9050646715235952,
 'cosine_mrr@10': 0.873971193415638,
 'cosine_map@100': 0.8739711934156378}

- 이제 results 디렉터리에 저장된 결과를 출력하면 다음과 같습니다.


In [21]:
# 결과 비교
df_st_original = pd.read_csv(
    "results/Information-Retrieval_evaluation_original_results.csv"
)
df_st_finetuned = pd.read_csv(
    "results/Information-Retrieval_evaluation_finetuned_results.csv"
)

df_st_original["model"] = "bge-m3"
df_st_finetuned["model"] = "fine_tuned"
df_st_all = pd.concat([df_st_original, df_st_finetuned])
df_st_all = df_st_all.set_index("model")

print("\n모델 성능 비교")
df_st_all


모델 성능 비교


,cosine_accuracy@1,cosine_accuracy@3,cosine_accuracy@5,cosine_accuracy@10,cosine_precision@1,cosine_precision@3,cosine_precision@5,cosine_precision@10,cosine_recall@1,cosine_recall@3,cosine_recall@5,cosine_recall@10,cosine_ndcg@10,cosine_mrr@10,cosine_map@100
model,,,,,,,,,,,,,,,
bge-m3,0.740741,0.907407,0.981481,1.0,0.740741,0.302469,0.196296,0.1,0.740741,0.907407,0.981481,1.0,0.871026,0.829189,0.829189
fine_tuned,0.796296,0.944444,0.962963,1.0,0.796296,0.314815,0.192593,0.1,0.796296,0.944444,0.962963,1.0,0.905065,0.873971,0.873971


- 파인튜닝 정 모델이 1.0으로 이미 최고 점수를 받은 경우에서는 동점을 기록했고, 나머지 모든 평가 지표에서는 파인튜닝 후 모델이 더 높은 성능을 나타냅니다.
- 이는 특정 도메인에 맞게 임베딩 모델을 파인튜닝하면 기존 모델보다 우수한 성능을 얻을 수 있음을 증명합니다.
